[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/01_track_A_enose_kesegaran.ipynb)

# 01 · Track A — Kesegaran Pangan dari *E-nose* Berbiaya Rendah

**Proyek ETS Machine Learning II — "Belajar dengan Label Mahal"**

Track ini menjadi proksi untuk skripsi Erlangga (MQ-135 + DHT22 + pH pada Raspberry Pi). Data pilot
Erlangga baru **6 baris dari 1 batch**, belum cukup untuk validasi. Karena itu kami memakai dataset
publik *e-nose* daging sapi (Wijaya dkk., 2018). Dataset ini juga memuat **MQ135**, kelembapan, dan suhu,
dengan 5 seri (5 potongan daging) yang berperan seperti `batch_id`.

| Aspek ML II | Bagian notebook |
|---|---|
| Data wrangling | §2 laporan kualitas data, §3 fitur kausal |
| Validasi yang jujur | §4 split acak vs *Leave-One-Series-Out* |
| Ensemble learning | §5 model tunggal vs bagging/boosting/stacking |
| Imbalanced data | §6 class weight, SMOTE, undersampling |
| Data augmentation | §7 jitter dan *gain scaling* sensor (khusus data latih) |
| Interpretabilitas | §8 SHAP per sensor |
| Data tambahan | §9 lebih banyak batch vs lebih banyak baris |

## 0. Instalasi & utilitas

In [ ]:
# Versi dikunci sama dengan uv.lock di repo supaya angka di Colab = angka di laporan
%pip install -q scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 optuna==5.0.0 \
    imbalanced-learn==0.14.2 rdkit==2026.9.1 shap==0.54.1

In [ ]:
import warnings, hashlib, json, time, urllib.request, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.base import clone
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              StackingClassifier, StackingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")
SEED = 42
RAW, RESULTS = Path("data/raw"), Path("results")
RAW.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(url, dest):
    import ssl, certifi                       # certifi: CA bundle yang sama di Colab dan macOS
    dest = Path(dest)
    if not dest.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "ets-ml2"})
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(req, context=ctx) as r, open(dest, "wb") as f:
            f.write(r.read())
    return dest

def mean_sd(v):
    v = np.asarray(v, float)
    return f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}"

## 1. Akuisisi data (unduh + verifikasi SHA-256)

In [ ]:
# Akuisisi: e-nose daging sapi (Wijaya dkk., 2018, Data in Brief 21) - Mendeley Data, CC BY 4.0
ENOSE_URL = ("https://data.mendeley.com/public-files/datasets/mwmhh766fc/files/"
             "2e65305f-5e53-4395-9b28-91dfc73ff490/file_downloaded")
ENOSE_SHA256 = "772bc2c75f2ac4b785538ea38311bdefaee65059b3391b080b4da5ea619963e3"
z = fetch(ENOSE_URL, RAW / "enose_beef.zip")
assert sha256(z) == ENOSE_SHA256, "checksum tidak cocok - file berubah/korup"
zipfile.ZipFile(z).extractall(RAW / "enose_beef")

CLASSES = ["excellent", "good", "acceptable", "spoiled"]      # ordinal: segar -> busuk
GAS = ["MQ135", "MQ136", "MQ2", "MQ3", "MQ4", "MQ5", "MQ6", "MQ8", "MQ9"]
PREDICTORS_RAW = GAS + ["Humidity"]
BASELINE_MIN, WINDOW = 10, 15

def load_enose():
    frames = []
    for f in sorted(glob.glob(str(RAW / "enose_beef" / "TS*.csv"))):
        d = pd.read_csv(f); d["series"] = Path(f).stem; frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    d["y"] = d["class"].map({c: i for i, c in enumerate(CLASSES)})
    return d

enose = load_enose()
print(enose.shape); enose.head()

## 2. Data wrangling — laporan kualitas per seri
Kami memeriksa nilai hilang, menit ganda, rentang suhu/kelembapan, dan distribusi kelas.
Label berasal dari jumlah mikroba (TVC). Kelasnya **tidak seimbang**: *spoiled* 39%, *excellent* 16%.

In [ ]:
report = []
for s, g in enose.groupby("series"):
    report.append({"series": s, "rows": len(g), "missing": int(g[PREDICTORS_RAW].isna().sum().sum()),
                   "duplicated_minutes": int(g["minute"].duplicated().sum()),
                   "temp_range_c": f"{g.Temperature.min():.1f}-{g.Temperature.max():.1f}",
                   **{c: int((g["class"] == c).sum()) for c in CLASSES}})
report = pd.DataFrame(report); display(report)
print(enose["class"].value_counts(normalize=True).round(3))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
for s, g in enose.groupby("series"):
    ax[0].plot(g.minute, g.MQ135, lw=.8, label=s); ax[1].plot(g.minute, g.MQ4, lw=.8, label=s)
ax[0].set_title("MQ135 per seri"); ax[1].set_title("MQ4 per seri"); ax[0].legend(); plt.show()

## 3. Rekayasa fitur kausal
Suhu disimpan sebagai **metadata**, bukan prediktor, sama seperti peran variabel di skripsi Erlangga.
Pengaruh menambahkan suhu diuji terpisah di §4.

In [ ]:
def engineer(g):
    """Fitur kausal satu seri: rasio terhadap baseline 10 menit pertama, rata-rata bergulir,
    dan kemiringan 15 menit. Semua hanya memakai data masa lalu (bisa dihitung real time)."""
    g = g.sort_values("minute")
    out = pd.DataFrame(index=g.index)
    for c in GAS:
        ratio = g[c] / g[c].iloc[:BASELINE_MIN].mean()
        out[f"{c}_ratio"] = ratio
        out[f"{c}_ratio_roll"] = ratio.rolling(WINDOW, min_periods=1).mean()
        out[f"{c}_slope"] = ratio.diff(WINDOW).fillna(0) / WINDOW
    out["Humidity"] = g["Humidity"]
    return out

def build_features(d, kind="engineered", with_temperature=False):
    X = d[PREDICTORS_RAW].copy() if kind == "raw" else \
        pd.concat([engineer(g) for _, g in d.groupby("series", sort=False)]).loc[d.index]
    if with_temperature:
        X["Temperature"] = d["Temperature"]
    return X

In [ ]:
X_demo = build_features(enose); print(X_demo.shape); X_demo.describe().T.head(8)

## 4. Validasi: split acak vs *Leave-One-Series-Out* (LOSO)
Menit yang bertetangga hampir identik. Split baris acak membuat model "menghafal" seri yang sama.
LOSO melatih model pada 4 seri dan menguji pada 1 seri yang belum pernah dilihat. Ini sama dengan
`GroupKFold` per `batch_id` di skripsi Erlangga.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold

def single_classifiers(cw=None):
    return {"LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight=cw)),
            "SVM-RBF": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", class_weight=cw, random_state=SEED)),
            "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
            "DecisionTree": DecisionTreeClassifier(max_depth=8, class_weight=cw, random_state=SEED)}

def ensemble_classifiers(cw=None):
    return {"RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1,
                                                   class_weight=cw, random_state=SEED),
            "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, n_jobs=4, random_state=SEED, verbosity=0),
            "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                           class_weight=cw, random_state=SEED, verbose=-1, n_jobs=4),
            "Stacking": StackingClassifier(
                estimators=[("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight=cw))),
                            ("svm", make_pipeline(StandardScaler(), SVC(C=10, probability=True, class_weight=cw,
                                                                         random_state=SEED))),
                            ("rf", RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1,
                                                          class_weight=cw, random_state=SEED))],
                final_estimator=LogisticRegression(max_iter=2000), cv=5, n_jobs=-1)}

def augment_series(d, copies, jitter, scale, rng):
    """Augmentasi deret waktu pada sinyal mentah: jitter = derau sensor,
    scale = variasi sensitivitas antar unit sensor MOS (gain ~ N(1, scale))."""
    out = []
    for s, g in d.groupby("series"):
        for k in range(copies):
            a = g.copy()
            for c in PREDICTORS_RAW:
                gain = rng.normal(1.0, scale) if scale > 0 else 1.0
                noise = rng.normal(0, jitter * g[c].std(), len(g)) if jitter > 0 else 0.0
                a[c] = g[c] * gain + noise
            a["series"] = f"{s}_aug{k}"; out.append(a)
    return pd.concat(out, ignore_index=True)

def scores(y, p):
    return {"macro_f1": f1_score(y, p, average="macro"), "accuracy": accuracy_score(y, p),
            "balanced_acc": balanced_accuracy_score(y, p),
            "recall_excellent": recall_score(y, p, labels=[0], average="macro", zero_division=0)}

def evaluate(model, d, split="loso", kind="engineered", with_temperature=False,
             resampler=None, augment=None, seed=SEED):
    """Seluruh prapemrosesan (augmentasi, resampling) di-fit HANYA pada fold latih."""
    X_all = build_features(d, kind, with_temperature)
    y_all, groups = d["y"].to_numpy(), d["series"].to_numpy()
    folds = (list(LeaveOneGroupOut().split(X_all, y_all, groups)) if split == "loso"
             else list(StratifiedKFold(5, shuffle=True, random_state=seed).split(X_all, y_all)))
    rng, rows = np.random.default_rng(seed), []
    for tr, te in folds:
        Xtr, ytr, gtr = X_all.iloc[tr], y_all[tr], groups[tr]
        if augment is not None:
            aug = augment_series(d.iloc[tr], rng=rng, **augment)
            Xtr = pd.concat([Xtr, build_features(aug, kind, with_temperature)], ignore_index=True)
            ytr = np.concatenate([ytr, aug["y"].to_numpy()])
            gtr = np.concatenate([gtr, aug["series"].str.split("_aug").str[0].to_numpy()])
        if resampler is not None:
            Xtr, ytr = clone(resampler).fit_resample(Xtr, ytr); gtr = None
        m = clone(model)
        if hasattr(m, "cv") and split == "loso" and gtr is not None:   # meta-learner stacking juga LOSO
            m.set_params(cv=list(LeaveOneGroupOut().split(np.zeros(len(gtr)), groups=gtr)))
        p = m.fit(Xtr, ytr).predict(X_all.iloc[te]).astype(int).ravel()
        rows.append({"fold": groups[te][0] if split == "loso" else len(rows), **scores(y_all[te], p)})
    return pd.DataFrame(rows)

SUMMARY = []
def run(tag, name, model, d=enose, label="", **kw):
    r = evaluate(model, d, **kw)
    SUMMARY.append({"experiment": tag, "model": name, "task": "binary" if d is not enose else "4-class",
                    "variant": label, "split": kw.get("split", "loso"), "features": kw.get("kind", "engineered"),
                    "macro_f1_mean": r.macro_f1.mean(), "macro_f1_sd": r.macro_f1.std(ddof=1)})
    print(f"{tag:12s} {name:13s} {SUMMARY[-1]['task']:7s} {SUMMARY[-1]['split']}/{SUMMARY[-1]['features']:10s} "
          f"{label:18s} macro-F1 = {mean_sd(r.macro_f1)}")
    return r

In [ ]:
lgbm, rf = ensemble_classifiers()["LightGBM"], ensemble_classifiers()["RandomForest"]
for split in ["random", "loso"]:
    for kind in ["raw", "engineered"]:
        run("E1_split", "LightGBM", lgbm, split=split, kind=kind)
        run("E1_split", "RandomForest", rf, split=split, kind=kind)
run("E1_temp", "LightGBM", lgbm, with_temperature=True, label="+Temperature")

**Bacaan:** split acak memberi macro-F1 ≈ 0,99, sedangkan LOSO hanya ≈ 0,5–0,6. Selisih ini adalah
*data leakage*. Angka ≈ 99% hanya bisa kami reproduksi dengan split baris acak.

## 5. Ensemble learning — model tunggal vs ensemble (LOSO)
Dua tugas: 4 kelas TVC dan **biner segar (excellent+good) vs tidak segar (acceptable+spoiled)**.
Tugas biner sama dengan label *Segar / Tidak segar* di skripsi Erlangga.

In [ ]:
binary = enose.assign(y=(enose["y"] >= 2).astype(int))
for d in [enose, binary]:
    for name, m in {**single_classifiers(), **ensemble_classifiers()}.items():
        run("E2_models", name, m, d=d)

## 6. Imbalanced data — class weight, SMOTE, undersampling (hanya pada fold latih)

In [ ]:
RESAMPLERS = {"SMOTE": SMOTE(random_state=SEED), "RandomUnder": RandomUnderSampler(random_state=SEED)}
for name in ["LogReg", "LightGBM"]:
    for cw in [None, "balanced"]:
        run("E3_imbalance", name, {**single_classifiers(cw), **ensemble_classifiers(cw)}[name],
            label=f"class_weight={cw}")
    for rname, rs in RESAMPLERS.items():
        run("E3_imbalance", name, {**single_classifiers(), **ensemble_classifiers()}[name],
            resampler=rs, label=rname)

## 7. Data augmentation deret waktu (hanya data latih, data uji tetap asli)

In [ ]:
AUGS = {"jitter": dict(copies=2, jitter=0.05, scale=0.0),
        "scaling": dict(copies=2, jitter=0.0, scale=0.10),
        "jitter+scaling": dict(copies=2, jitter=0.05, scale=0.10),
        "jitter+scaling x4": dict(copies=4, jitter=0.05, scale=0.10)}
for name in ["LogReg", "LightGBM"]:
    m = {**single_classifiers(), **ensemble_classifiers()}[name]
    run("E4_augment", name, m, label="none")
    for aname, a in AUGS.items():
        run("E4_augment", name, m, augment=a, label=aname)
summary = pd.DataFrame(SUMMARY); summary.to_csv(RESULTS / "enose_summary.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
s = summary[(summary.experiment == "E2_models") & (summary.task == "binary")]
ax[0].barh(s.model, s.macro_f1_mean, xerr=s.macro_f1_sd, color=["#9aa5a6"] * 4 + ["#0f5257"] * 4)
ax[0].set_title("Biner segar vs tidak segar - LOSO macro-F1"); ax[0].set_xlim(0, 1)
a = summary[summary.experiment == "E4_augment"].pivot(index="variant", columns="model", values="macro_f1_mean")
a.plot.barh(ax=ax[1]); ax[1].set_title("Augmentasi - LOSO macro-F1 (4 kelas)"); plt.tight_layout(); plt.show()

## 8. Interpretabilitas — sensor mana yang menentukan keputusan segar/tidak segar?

In [ ]:
import shap
X = build_features(enose); yb = binary["y"].to_numpy(); groups = enose["series"].to_numpy()
parts = []
for tr, te in LeaveOneGroupOut().split(X, yb, groups):
    m = clone(ensemble_classifiers()["XGBoost"]).fit(X.iloc[tr], yb[tr])
    parts.append(pd.Series(np.abs(shap.TreeExplainer(m).shap_values(X.iloc[te])).mean(0), index=X.columns))
feat = pd.concat(parts, axis=1).mean(1)
sensor = feat.groupby(feat.index.str.split("_").str[0]).sum().sort_values()
sensor.plot.barh(title="Mean |SHAP| per sensor (XGBoost, dijelaskan pada seri uji LOSO)", figsize=(7, 4)); plt.show()
sensor.sort_values(ascending=False).round(3)

## 9. Data tambahan: lebih banyak *batch* atau lebih banyak baris?
Pertanyaan ini menjawab bagian rubrik *"bagaimana tambahan data meningkatkan performa"*,
dan langsung relevan dengan rancangan 16 batch di skripsi Erlangga.

In [ ]:
from itertools import combinations
rng, rows, names = np.random.default_rng(SEED), [], sorted(enose.series.unique())
xgb = ensemble_classifiers()["XGBoost"]; series = enose.series.to_numpy()
for test_s in names:
    te = series == test_s; others = [s for s in names if s != test_s]; all_tr = np.flatnonzero(~te)
    for k in range(1, 5):
        for combo in combinations(others, k):
            tr = np.flatnonzero(np.isin(series, combo))
            p = clone(xgb).fit(X.iloc[tr], yb[tr]).predict(X[te])
            rows.append({"design": "lebih banyak seri", "n_series": k, "n_rows": len(tr),
                         "macro_f1": f1_score(yb[te], p, average="macro")})
    for n in [40, 160, 640]:
        tr = rng.choice(all_tr, n, replace=False)
        p = clone(xgb).fit(X.iloc[tr], yb[tr]).predict(X[te])
        rows.append({"design": f"{n} baris dari 4 seri", "n_series": 4, "n_rows": n,
                     "macro_f1": f1_score(yb[te], p, average="macro")})
curve = pd.DataFrame(rows)
curve.groupby(["design", "n_series"]).macro_f1.agg(["mean", "std", "count"]).round(3)

**Bacaan:** macro-F1 naik dari ≈0,72 (1 seri) ke ≈0,91 (4 seri). Sebaliknya, 160 baris yang diambil
dari 4 seri saja sudah ≈0,87. **Menambah batch lebih berharga daripada menambah baris.**